# Tiny Black-and-White Text-to-Pixel AI
## The simplest version: **one word → one tiny image**

We will teach a neural network to generate 16×16 black-and-white pixel art from five words:

`heart` · `face` · `robot` · `tree` · `spaceship`

The central idea is:

**WORD + RANDOMNESS → DECODER → IMAGE**


## 1. Create the training images

We use 5,000 images: **1,000 slightly different examples for each word**.
The next cell downloads our tiny dataset generator and creates the images automatically.


In [ ]:
# Download the dataset generator from GitHub.
!wget -q -O generate_dataset.py https://raw.githubusercontent.com/dujing82-blip/tiny-black-white-generative-ai/main/generate_dataset.py

# Remove an old copy if this cell has been run before.
!rm -rf pixel_dataset

# Generate 5,000 black-and-white 16x16 images.
!python generate_dataset.py


## 2. Load the dataset

Each image contains only 256 pixels: 16 × 16.
Black = 0. White = 1.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from PIL import Image

# Read the table that tells us which word belongs to each image.
table = pd.read_csv('pixel_dataset/labels.csv')

# Load every PNG image and convert pixel values from 0-255 to 0-1.
images = []
for filename in table['filename']:
    img = Image.open(os.path.join('pixel_dataset', filename)).convert('L')
    images.append(np.array(img, dtype=np.float32) / 255.0)

X = np.stack(images)       # Shape: (5000, 16, 16)
y = table['label'].to_numpy(np.int32)  # heart=0, face=1, ...

words = ['heart', 'face', 'robot', 'tree', 'spaceship']
print('Images:', X.shape)
print('Labels:', y.shape)


### Look at a few training images


In [ ]:
plt.figure(figsize=(10, 2))
for i, word in enumerate(words):
    index = np.where(y == i)[0][0]
    plt.subplot(1, 5, i + 1)
    plt.imshow(X[index], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    plt.title(word)
    plt.axis('off')
plt.show()


## 3. The Encoder

The encoder looks at a training image and its word and compresses them into a small **latent representation** called `z`.

We use only **4 latent numbers**.


In [ ]:
# Number of values in our tiny latent space.
LATENT_DIM = 4

# ----- INPUTS -----
# The image input is 16x16 pixels.
image_input = keras.Input(shape=(16, 16), name='image')

# The word is represented by one integer: 0, 1, 2, 3, or 4.
word_input = keras.Input(shape=(), dtype='int32', name='word')

# ----- IMAGE PATH -----
# Flatten changes the 16x16 image into a list of 256 numbers.
x = layers.Flatten()(image_input)

# ----- WORD PATH -----
# Embedding turns the word ID into a small learned vector.
# This is the first important idea: WORD -> NUMERICAL REPRESENTATION.
word_vector = layers.Embedding(input_dim=5, output_dim=4)(word_input)
word_vector = layers.Flatten()(word_vector)

# Put the image information and word information together.
x = layers.Concatenate()([x, word_vector])

# Two ordinary neural-network layers.
x = layers.Dense(64, activation='relu')(x)
x = layers.Dense(32, activation='relu')(x)

# A VAE does not encode an image as one exact point.
# It learns a mean and a log-variance for a probability distribution.
z_mean = layers.Dense(LATENT_DIM, name='z_mean')(x)
z_log_var = layers.Dense(LATENT_DIM, name='z_log_var')(x)

# Sampling chooses a random point z from that distribution.
class Sampling(layers.Layer):
    def call(self, inputs):
        mean, log_var = inputs
        random_noise = tf.random.normal(shape=tf.shape(mean))
        return mean + tf.exp(0.5 * log_var) * random_noise

z = Sampling()([z_mean, z_log_var])

# The encoder is now a complete Keras model.
encoder = keras.Model(
    [image_input, word_input],
    [z_mean, z_log_var, z],
    name='encoder'
)

encoder.summary()


## 4. The Decoder — this is our image generator

The decoder receives two things:

1. the word — **what should I draw?**
2. random latent vector `z` — **what variation should this example have?**

Then it outputs 256 pixel values.


In [ ]:
# ----- INPUTS -----
# z contains four numbers describing random variation.
z_input = keras.Input(shape=(LATENT_DIM,), name='z')

# Again, the word is one integer from 0 to 4.
decoder_word_input = keras.Input(shape=(), dtype='int32', name='word')

# Turn the word into a learned numerical vector.
decoder_word_vector = layers.Embedding(input_dim=5, output_dim=4)(decoder_word_input)
decoder_word_vector = layers.Flatten()(decoder_word_vector)

# Combine WHAT to draw (word) with HOW it varies (z).
x = layers.Concatenate()([z_input, decoder_word_vector])

# Ordinary neural-network layers turn those numbers into pixels.
x = layers.Dense(32, activation='relu')(x)
x = layers.Dense(64, activation='relu')(x)

# We need 16 x 16 = 256 output pixels.
# Sigmoid keeps every pixel value between 0 and 1.
x = layers.Dense(256, activation='sigmoid')(x)

# Change the list of 256 numbers back into a 16x16 image.
output_image = layers.Reshape((16, 16))(x)

decoder = keras.Model(
    [z_input, decoder_word_input],
    output_image,
    name='decoder'
)

decoder.summary()


## 5. Train the VAE

Training asks the model to reconstruct its training images.

The loss has two parts:

- **reconstruction loss:** Does the generated image look like the original?
- **KL loss:** Keep the latent space smooth enough that we can later sample random points from it.


In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        # data contains a batch of images and their word IDs.
        images, word_ids = data

        with tf.GradientTape() as tape:
            # 1. Encode each training image into latent z.
            z_mean, z_log_var, z = self.encoder([images, word_ids], training=True)

            # 2. Decode z + word back into an image.
            reconstructed = self.decoder([z, word_ids], training=True)

            # 3. Reconstruction loss: pixel-by-pixel squared error.
            reconstruction_loss = tf.reduce_mean(
                tf.reduce_sum(tf.square(images - reconstructed), axis=[1, 2])
            )

            # 4. KL loss organizes the latent space.
            kl_loss = -0.5 * tf.reduce_mean(
                tf.reduce_sum(
                    1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var),
                    axis=1
                )
            )

            # A small KL weight keeps reconstruction easy for this classroom demo.
            total_loss = reconstruction_loss + 0.02 * kl_loss

        # Standard neural-network learning: compute gradients and update weights.
        gradients = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))

        return {
            'loss': total_loss,
            'reconstruction': reconstruction_loss,
            'KL': kl_loss
        }

# Make batches of 64 examples and shuffle them.
dataset = tf.data.Dataset.from_tensor_slices((X, y))
dataset = dataset.shuffle(len(X)).batch(64)

# Create and train the model.
vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001))

# 50 epochs is still small enough for Colab, but gives the simple model time to learn.
history = vae.fit(dataset, epochs=50)


## 6. Generate brand-new images

Now we no longer need the encoder.

We simply choose a word, create random `z`, and ask the **decoder** to draw.


In [ ]:
# Convert words to the integer IDs used during training.
word_to_id = {word: i for i, word in enumerate(words)}

def generate(word, number=10):
    # Check that the word is one of our five known words.
    if word not in word_to_id:
        print('Choose:', words)
        return

    # Create a DIFFERENT random z for every new image.
    # This randomness lets one word generate multiple variations.
    random_z = np.random.normal(size=(number, LATENT_DIM)).astype('float32')

    # Repeat the chosen word ID once for each image.
    word_ids = np.full(number, word_to_id[word], dtype=np.int32)

    # WORD + RANDOM z -> DECODER -> IMAGE
    generated = decoder.predict([random_z, word_ids], verbose=0)

    # Display the generated images.
    plt.figure(figsize=(15, 2))
    for i in range(number):
        plt.subplot(1, number, i + 1)
        plt.imshow(generated[i], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        plt.axis('off')
    plt.suptitle('Prompt: ' + word, fontsize=16)
    plt.show()

# Try it!
generate('robot')


## 7. Same word, different random `z`

Run this cell several times. Every time, the word stays `spaceship`, but `z` changes.


In [ ]:
generate('spaceship')


## 8. Tiny interactive text-to-image app


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

text_box = widgets.Text(value='robot', description='Word:')
button = widgets.Button(description='Generate')
output = widgets.Output()

def on_click(_):
    with output:
        clear_output(wait=True)
        generate(text_box.value.lower().strip())

button.on_click(on_click)
display(text_box, button, output)


# The whole lesson in one picture

```text
            WHAT?                    VARIATION
              |                          |
           "robot"                  random z
              |                          |
          Embedding                      |
              \                         /
               \                       /
                +---------------------+
                          |
                       DECODER
                          |
                          v
                    16 x 16 IMAGE
```

### Three ideas to remember

1. **Embedding:** turns a word into numbers a neural network can learn from.
2. **Latent z:** supplies variation/randomness.
3. **Decoder:** turns those numbers into pixels.

That is our tiny generative AI.
